# 02 - Clustering


In [1]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

df = pd.read_csv('dataset_timnas.csv')
print(df.shape)
df.head()

(635, 8)


,text,created_at,like_count,repost_count,reply_count,url,kata_kunci,text_clean
0,@SirSimbolon @laazuardii @xcordeirro ibarat ca...,2026-10-11 15:57:12,0,0,0,https://x.com/sekusipapa/status/21092066733448...,garuda,ibarat capo ultras garuda tahun 98
1,@besibakar @ahlireparasi @MusuhMisuh2 @LambeSa...,2026-10-11 15:50:19,0,0,0,https://x.com/sekologic/status/210920493830814...,garuda,stelah era mbg revitalisasi skolah makin bnyak...
2,The #Garuda #Vishnu statue visible for miles i...,2026-10-11 15:50:11,0,0,0,https://x.com/anirusan/status/2109204906423066851,garuda,the garuda vishnu statue visible for miles in ...
3,"zonauang 💞\r\nsejak '22, 35OO++ testii bio🥰\r\...",2026-10-11 15:48:51,0,0,0,https://x.com/skripsijember/status/21092045703...,garuda,zonauang sejak 22 35oo testii bio publish jurn...
4,@txtruangpodcast samaa casenya kaya alat makan...,2026-10-11 15:44:39,3,0,0,https://x.com/preitzle/status/2109203515814076900,garuda,samaa casenya kaya alat makan amp piring garud...


In [2]:
df['text_clean'] = df['text_clean'].fillna('').astype(str).str.strip()
dfc = df[df['text_clean'] != ''].reset_index(drop=True)
print('Baris dipakai:', len(dfc), '| dikeluarkan:', len(df) - len(dfc))

Baris dipakai: 635 | dikeluarkan: 0


In [3]:
stop_id = {'yang','dan','di','ke','dari','ini','itu','untuk','dengan','pada','adalah','atau','juga','karena','dalam','akan','sudah','udah','belum','ada','saya','aku','kami','kita','kamu','dia','mereka','nya','lah','kah','pun','saja','aja','lagi','masih','bisa','dapat','oleh','sebagai','tapi','tetapi','namun','jadi','kalau','kalo','agar','supaya','para','tersebut','sama','hanya','cuma','banget','sih','dong','deh','kok','yah','ya','yg','dgn','utk','dr','krn','sdh','udh','tdk','jd','sm','klo','gue','gua','lo','lu','emang','memang','mau','lebih','sangat','semua','setiap','setelah','sebelum','saat','ketika','baru'}
stop_slang = {'wkwk','wkwkwk','wkwkwkwk','haha','hahaha','hehe','anjir','anjay','njir','bgt','gak','ga','nggak','ngga','gk','kan','nih','tuh','mah','dah','lho','loh','nah','btw','eh','oh','ok','oke','rt','amp'}
stopwords = sorted(stop_id | stop_slang | set(ENGLISH_STOP_WORDS))

In [5]:
tfidf = TfidfVectorizer(stop_words=stopwords, max_df=0.9, min_df=2)
X = tfidf.fit_transform(dfc['text_clean'])
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
dfc['cluster'] = kmeans.fit_predict(X)
terms = tfidf.get_feature_names_out()

for k in range(3):
    top5 = [terms[i] for i in kmeans.cluster_centers_[k].argsort()[::-1][:5]]
    sub = dfc[dfc['cluster'] == k]
    print(f'\n Cluster {k} | {len(sub)} postingan ')
    print('5 kata dominan:', top5)
    for t in sub['text'].head(3):
        print(' -', t)


 Cluster 0 | 83 postingan 
5 kata dominan: ['garuda', 'jembatan', 'like', 'pembangunan', 'lk']
 - @SirSimbolon @laazuardii @xcordeirro ibarat capo ultras garuda tahun 98
 - @besibakar @ahlireparasi @MusuhMisuh2 @LambeSahamjja Stelah era MBG revitalisasi skolah makin bnyak kok, ada sekolah gratis, sekolah garuda , sekolah rakyat ada.
 - The #Garuda #Vishnu statue visible for miles in #Bali is a sight to behold! #Hinduism https://t.co/QOLkylJzgu

 Cluster 1 | 438 postingan 
5 kata dominan: ['timnas', 'indonesia', 'pssi', 'pemain', 'fifa']
 - zonauang 💞
sejak '22, 35OO++ testii bio🥰
publish jurnal sinta, publish jurnal non sinta garuda &amp; jurnal inter
joki skripsi, joki tesis, revisi skripsi, revisi tesis, joki jurnal artikel, revisi jurnal, artikel, joki tugas, ppt, laporan, ppt, essay, joki olah data spss etc♡
 - @SubTahu @nameles_id @Icaljsap Sebenarnya yg ingin Full lokal itu adalah para Mafia bola negara seberang dan antek2nya di Indo,  yg gak ingin sepakbola Indo maju. Mereka ga